# Uber Entry and Alcohol-Related Traffic Fatalities
Replication of Greenwood & Wattal (2017)


In this project, I’m replicating the main difference-in-differences analysis from Greenwood & Wattal (2017).  
I'm looking at whether Uber entering a city reduces alcohol-related traffic fatalities.  

**Author:** Hannah Brannam  
**Date:** Spring 2026

## 1. Introduction
Drunk driving remains a serious public safety issue in the United States. Each year, alcohol-impaired driving is responsible for thousands of traffic fatalities and creates large social costs, including lost lives, medical expenses, and broader economic impacts. Because driving under the influence puts others at risk, policymakers have long looked for ways to reduce impaired driving through regulation, enforcement, and public awareness efforts.

The rise of ride-sharing platforms such as Uber introduced a new potential way to address this problem. By making it much easier to get a ride on demand, ride-sharing services may give people a convenient alternative to driving after drinking. In theory, when transportation becomes cheaper and more accessible, individuals who might otherwise drive home after a night out could choose a ride-share instead. This effect may be particularly strong in urban areas where ride-sharing networks are dense and wait times are short.

At the same time, the overall impact of ride-sharing on traffic safety is not necessarily straightforward. Ride-sharing services could increase the total amount of travel on the road, encourage more nightlife activity, or mainly replace taxi rides rather than private driving. If that is the case, the net effect on alcohol-related traffic fatalities may be small or unclear. For this reason, the relationship between ride-sharing and traffic safety is ultimately an empirical question.

In an influential study, Matthew Greenwood and Sunil Wattal (2017) examine whether Uber’s entry into cities reduced alcohol-related traffic fatalities using a difference-in-differences research design. Their results suggest that Uber’s arrival is associated with reductions in alcohol-related crashes in some urban areas.

This project replicates the core empirical strategy of that study using county-level data from California between 2010 and 2014. The analysis combines fatal accident data from the Fatality Analysis Reporting System (FARS), county population estimates from the United States Census Bureau, and information on Uber launch dates across counties.

The central research question is straightforward: Does Uber’s entry into a county reduce alcohol-related traffic fatalities? By applying the same difference-in-differences framework used in the original study, this project evaluates whether similar patterns appear in California counties during the early years of Uber’s expansion.

In [1]:
# Core libraries
import pandas as pd
import numpy as np

# Regressions
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

## 2. Literature Review

This study builds on a growing body of research that looks at how transportation access affects traffic safety. Traditionally, policies aimed at reducing drunk driving have focused on legal enforcement, such as stricter DUI penalties, sobriety checkpoints, and public awareness campaigns. More recently, researchers have started to explore whether changes in transportation technology, like ride-sharing platforms, might also influence risky driving behavior.

One of the earliest and most well-known studies on this topic is by Matthew Greenwood and Sunil Wattal (2017). In their paper Show Me the Way to Go Home: An Empirical Investigation of Ride Sharing and Alcohol Related Motor Vehicle Homicide, the authors examine whether the introduction of Uber affected alcohol-related traffic fatalities across U.S. cities. Using panel data and a difference-in-differences approach, they compare changes in fatality rates before and after Uber entered a city to cities where Uber had not yet launched. Their results suggest that Uber’s entry is associated with statistically significant reductions in alcohol-related traffic fatalities in some urban areas.

The basic economic intuition behind their findings is fairly straightforward. If ride-sharing makes it cheaper and easier to find a ride home, people who might otherwise drive after drinking may choose to take an Uber instead. In that sense, ride-sharing provides an alternative to driving and could reduce risky behavior by lowering the cost of not driving.

At the same time, the relationship between ride-sharing and traffic safety is not necessarily simple. Ride-sharing services might increase total driving on the road, replace public transportation trips, or substitute mainly for taxis rather than private cars. If that is the case, the overall impact on alcohol-related fatalities could be small or unclear. The effects may also differ depending on factors like population density, urban infrastructure, and local drinking patterns.

By replicating the core difference-in-differences design used by Greenwood and Wattal (2017) with county-level data from California, this project explores whether similar patterns appear within a single state during the early years of Uber’s expansion.

## 3. Identification Strategy

To estimate the causal effect of Uber entry on alcohol-related traffic fatalities, I use a difference-in-differences (DiD) research design. The key idea of this approach is to compare changes in outcomes over time between counties that experience Uber entry and counties that have not yet experienced entry.

Uber did not enter all California counties at the same time. Instead, its rollout occurred in a staggered manner across major urban areas. This staggered timing provides quasi-experimental variation that can be used to estimate the impact of Uber availability on fatality rates.

The identifying variation comes from comparing:
1. Counties before and after Uber entry
2. Relative to counties that have not yet experienced Uber entry in the same year
   
This approach allows me to difference out both time-invariant county characteristics and statewide year-specific shocks.

### 3.1 Econometric Specification

The empirical analysis follows a difference-in-differences framework similar to Greenwood & Wattal (2017). The baseline regression model is:

            FatalityRatect​=β0​+β1​UberActivect​+γc​+δt​+εct​

where:

* c indexes counties
* t indexes years
* FatalityRate(ct) is the alcohol-related traffic fatality rate per 100,000 residents
* UberActive(ct) equals 1 if Uber has entered county (c) by year (t), and 0 otherwise
* γc represents county fixed effects
* δtrepresents year fixed effects
* εct is the error term

The coefficient of interest is β1. It measures the average change in alcohol-related fatality rates after Uber becomes active in a county, relative to counties where Uber has not yet entered.

### 3.2 Role of Fixed Effects

Including county fixed effects controls for all time-invariant differences across counties. These may include:

* Urbanization levels
* Baseline drinking culture
* Road infrastructure quality
* Long-standing enforcement intensity
* Demographic composition

By including county fixed effects, the model compares each county to itself over time.

Year fixed effects (( \delta_t )) control for statewide or national shocks that affect all counties in a given year. Examples include:

* Changes in DUI enforcement laws
* Economic fluctuations
* Gas prices
* Statewide safety campaigns

Together, these fixed effects ensure that identification comes from within-county changes over time, net of common time shocks.

### 3.3 The Parallel Trends Assumption

The key identifying assumption in a difference-in-differences design is the parallel trends assumption.

Formally, in the absence of Uber entry, treated and untreated counties would have experienced similar trends in alcohol-related fatality rates over time.

This assumption cannot be tested directly, since we do not observe the counterfactual outcome. However, its plausibility can be partially evaluated by:

* Comparing pre-treatment trends between treated and untreated counties
* Examining graphical patterns prior to Uber entry
* Checking robustness to alternative specifications

If treated counties were already experiencing declining fatality rates prior to Uber entry, the DiD estimate could mistakenly attribute that pre-existing trend to Uber.

### 3.4 Staggered Adoption Considerations

Uber’s entry occurred at different times across counties. In a staggered adoption setting, counties serve as controls for each other at different points in time. Counties that adopt Uber later act as control units for early adopters in earlier years.

This staggered structure increases the amount of identifying variation but also requires careful interpretation. The coefficient represents an average treatment effect across counties and years, potentially masking heterogeneous effects across urban and rural areas.


### 3.5 Standard Errors

Standard errors are clustered at the county level. Clustering accounts for potential serial correlation in fatality rates within counties over time. Failure to cluster in panel data settings can lead to underestimated standard errors and over-rejection of the null hypothesis.


## 4. Data
The dataset used in this analysis combines three sources:

1. Fatal crash data from the Fatality Analysis Reporting System (FARS)
2. County population estimates from the U.S. Census Bureau
3. Data on Uber entry dates across California counties

The final dataset is organized at the county-year level.

### 4.1 Fatality Data (FARS)
Traffic fatality data come from the Fatality Analysis Reporting System (FARS), a nationwide census of fatal motor vehicle crashes maintained by the National Highway Traffic Safety Administration (NHTSA). FARS includes detailed information about every crash in the United States that results in at least one fatality.

For this analysis, I extract accident-level data from the FARS dataset for the years 2009 through 2014 and restrict the sample to crashes occurring in California. Alcohol involvement is identified using the DRUNK_DR variable, which records whether at least one driver involved in the crash was legally intoxicated.

The data are then aggregated to the county-year level, producing counts of total fatalities and alcohol-related fatalities for each county in each year.

In [2]:
years = range(2009, 2015)
all_years = []

for year in years:
    
    df_year = pd.read_csv(f"data/raw/FARS{year}NationalCSV/accident.csv")
    
    # Filter California
    ca = df_year[df_year['STATE'] == 6].copy()
    
    # Alcohol involvement
    ca['alcohol_crash'] = (ca['DRUNK_DR'] > 0).astype(int)
    ca['alcohol_fatalities'] = ca['FATALS'] * ca['alcohol_crash']
    
    # Aggregate to county × year
    county_year = ca.groupby('COUNTY').agg({
        'FATALS': 'sum',
        'alcohol_fatalities': 'sum'
    }).reset_index()
    
    county_year['YEAR'] = year
    
    all_years.append(county_year)

ca_panel = pd.concat(all_years, ignore_index=True)

ca_panel.head()


,COUNTY,FATALS,alcohol_fatalities,YEAR
0,1,71,30,2009
1,3,1,1,2009
2,5,13,2,2009
3,7,23,12,2009
4,9,12,4,2009


### 4.2 Population Data
County population estimates are obtained from the U.S. Census Bureau’s County Population Estimates dataset. These estimates provide annual population counts for all U.S. counties.

The population data are reshaped into a long format so they can be merged with the county-year fatality dataset. Population counts are necessary to construct a fatality rate, allowing comparisons across counties of different sizes.

In [3]:
pop_raw = pd.read_csv("data/raw/CO-EST2020-alldata.csv", encoding="latin1")

# Keep California counties only
pop_ca = pop_raw[(pop_raw['SUMLEV'] == 50) & (pop_raw['STATE'] == 6)].copy()

pop_ca = pop_ca[['STATE','COUNTY',
                 'POPESTIMATE2010',
                 'POPESTIMATE2011',
                 'POPESTIMATE2012',
                 'POPESTIMATE2013',
                 'POPESTIMATE2014']]

# Reshape to long format
pop_long = pop_ca.melt(
    id_vars=['STATE','COUNTY'],
    var_name='YEAR',
    value_name='population'
)

pop_long['YEAR'] = pop_long['YEAR'].str[-4:].astype(int)

pop_long.head()

,STATE,COUNTY,YEAR,population
0,6,1,2010,1512997
1,6,3,2010,1161
2,6,5,2010,37883
3,6,7,2010,219951
4,6,9,2010,45467


### 4.3 Merging Population and Crash Data
The population data are reshaped and merged with the crash dataset to create a county-year panel.

In [13]:
df = ca_panel.merge(
    pop_long[['COUNTY','YEAR','population']],
    on=['COUNTY','YEAR'],
    how='left'
)

# Drop 2009 
df = df[df['YEAR'] >= 2010].copy()

df.head()

,COUNTY,FATALS,alcohol_fatalities,YEAR,population
57,1,63,18,2010,1512997.0
58,3,1,0,2010,1161.0
59,5,7,1,2010,37883.0
60,7,46,20,2010,219951.0
61,9,12,4,2010,45467.0


### 4.4 Uber Entry Data

Information on when Uber entered each county was collected from publicly available sources describing the company’s expansion across California during the early 2010s. These sources provide approximate entry years for when Uber first began operating in different local markets.

Using this information, I create a variable called uber_active, which equals 1 for county–year observations after Uber has entered a county and 0 otherwise. This variable captures whether ride-sharing services were available in a given county in a particular year.

Because Uber did not enter all counties at the same time, different counties are treated at different points in time. This staggered rollout provides useful variation for the difference-in-differences analysis

In [5]:
uber_entry = pd.read_csv("data/raw/uber_entry_ca.csv")

df = df.merge(uber_entry, on='COUNTY', how='left')

# Treatment indicator
df['uber_active'] = (df['YEAR'] >= df['uber_entry_year']).astype(int)

df.head()

,COUNTY,FATALS,alcohol_fatalities,YEAR,population,uber_entry_year,uber_active
0,1,63,18,2010,1512997.0,NaN,0
1,3,1,0,2010,1161.0,NaN,0
2,5,7,1,2010,37883.0,NaN,0
3,7,46,20,2010,219951.0,NaN,0
4,9,12,4,2010,45467.0,NaN,0


### 4.5 Constructing the Fatality Rate
To make fatality counts comparable across counties of different population sizes, I construct an alcohol-related fatality rate per 100,000 residents.

This is calculated as:

Fatality Rate = (Alcohol Fatalities / Population) × 100,000

Using rates rather than raw counts ensures that changes in fatalities are interpreted relative to the size of the county population.

In [6]:
df['fatality_rate'] = (df['alcohol_fatalities'] / df['population']) * 100000

df[['COUNTY','YEAR','fatality_rate']].head()

,COUNTY,YEAR,fatality_rate
0,1,2010,1.189692
1,3,2010,0.000000
2,5,2010,2.639706
3,7,2010,9.092934
4,9,2010,8.797589


### 4.6 Descriptive Statistics

Before estimating the regression model, it is useful to examine summary statistics for the key variables in the dataset. These statistics provide a basic overview of fatality rates, county populations, and Uber entry across the sample.

In [10]:
df[['fatality_rate','alcohol_fatalities','population','uber_active']].describe()

,fatality_rate,alcohol_fatalities,population,uber_active
count,288.000000,288.000000,2.880000e+02,288.000000
mean,5.254382,15.520833,6.587487e+05,0.041667
std,8.409374,26.531152,1.431612e+06,0.200174
min,0.000000,0.000000,1.083000e+03,0.000000
25%,1.637770,2.000000,4.539000e+04,0.000000
50%,3.286327,6.000000,1.807880e+05,0.000000
75%,5.731608,15.000000,7.000408e+05,0.000000
max,88.417330,181.000000,1.003345e+07,1.000000


## 5. Empirical Strategy
The baseline model estimates the relationship between Uber entry and alcohol-related fatality rates using a difference-in-differences specification with county and year fixed effects.

### 5.1 Baseline Regression
The baseline regression estimates the relationship between Uber entry and alcohol-related fatality rates using county and year fixed effects.

In [7]:
model = smf.ols(
    'fatality_rate ~ uber_active + C(COUNTY) + C(YEAR)',
    data=df
).fit(cov_type='cluster', cov_kwds={'groups': df['COUNTY']})

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:          fatality_rate   R-squared:                       0.399
Model:                            OLS   Adj. R-squared:                  0.234
Method:                 Least Squares   F-statistic:                     63.83
Date:                Sun, 08 Mar 2026   Prob (F-statistic):           4.24e-22
Time:                        20:02:17   Log-Likelihood:                -948.00
No. Observations:                 288   AIC:                             2022.
Df Residuals:                     225   BIC:                             2253.
Df Model:                          62                                         
Covariance Type:              cluster                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept            0.0940      0.659  

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 62, but rank is 5
  warnings.warn('covariance of constraints does not have full '


### 5.2 Interpretation

The estimated coefficient on uber_active is -0.59, implying that Uber entry is associated with a decrease of roughly 0.59 alcohol-related fatalities per 100,000 residents.

However, the estimate is not statistically significant (p = 0.471). This means that within this dataset, we cannot reject the null hypothesis that Uber entry has no effect on alcohol-related fatality rates.

While the coefficient is negative and therefore consistent with the hypothesis that ride-sharing services may reduce drunk driving, the lack of statistical significance suggests the evidence is weak in this sample.

## 6. Robustness Checks

To examine whether Uber’s impact may differ across geographic areas, I estimate the model using only counties with above-median populations. Ride-sharing services are more commonly used in urban areas, so focusing on larger counties may provide a clearer test of the hypothesis.

### 6.1 Larger Counties Only

The sample is restricted to counties with populations above the median population level in the dataset.

In [11]:
median_pop = df['population'].median()

df_large = df[df['population'] > median_pop]

large_model = smf.ols(
    'fatality_rate ~ uber_active + C(COUNTY) + C(YEAR)',
    data=df_large
).fit(cov_type='cluster', cov_kwds={'groups': df_large['COUNTY']})

print(large_model.summary())

                            OLS Regression Results                            
Dep. Variable:          fatality_rate   R-squared:                       0.618
Model:                            OLS   Adj. R-squared:                  0.503
Method:                 Least Squares   F-statistic:                     8.002
Date:                Mon, 09 Mar 2026   Prob (F-statistic):           8.70e-05
Time:                        00:43:56   Log-Likelihood:                -205.96
No. Observations:                 144   AIC:                             479.9
Df Residuals:                     110   BIC:                             580.9
Df Model:                          33                                         
Covariance Type:              cluster                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept            1.4966      0.302  

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 33, but rank is 5
  warnings.warn('covariance of constraints does not have full '


### 6.2 Log Transformation of Fatality Rate

As an additional robustness check, I estimate the model using the logarithm of the fatality rate as the dependent variable. Log transformations are commonly used in empirical work to reduce the influence of extreme values and to interpret coefficients in percentage terms.

Because some counties may have very small fatality rates, a small constant (0.01) is added before taking the logarithm.

In [1]:
df['log_fatality_rate'] = np.log(df['fatality_rate'] + 0.01)

log_model = smf.ols(
    'log_fatality_rate ~ uber_active + C(COUNTY) + C(YEAR)',
    data=df
).fit(cov_type='cluster', cov_kwds={'groups': df['COUNTY']})

print(log_model.summary())

NameError: name 'np' is not defined

### 6.3 Interpretation

The coefficient on uber_active in the log specification represents the approximate percentage change in alcohol-related fatality rates after Uber entry.

The estimated coefficient remains statistically insignificant, suggesting that the results are not sensitive to whether the dependent variable is measured in levels or logarithms. This strengthens the conclusion that Uber entry does not appear to have a statistically detectable effect on alcohol-related fatality rates in this dataset.

## 7. Conclusion
This project replicated the empirical strategy of Greenwood & Wattal (2017) to examine whether Uber entry affects alcohol-related traffic fatalities. Using county-level data from California between 2010 and 2014, I estimated a difference-in-differences model with county and year fixed effects.

The results suggest that Uber entry is associated with a small reduction in alcohol-related fatality rates. However, the estimated effects are not statistically significant in this dataset. Robustness checks focusing on larger counties produce similar results.

There are several possible explanations for these findings. The time period examined may be too short to fully capture Uber’s expansion, and county-level data may mask variation within cities where ride-sharing usage is concentrated.

Future research could extend this analysis by examining longer time periods, incorporating additional 
states, or exploring heterogeneous effects across urban and rural areas.

Overall, this replication illustrates how difference-in-differences methods can be used to study the real-world impact of technological changes in transportation markets.

## References

Greenwood, B. N., & Wattal, S. (2017). *Show me the way to go home: An empirical investigation of ride sharing and alcohol-related motor vehicle fatalities.* MIS Quarterly, 41(1), 163–187.

Angrist, J. D., & Pischke, J.-S. (2009). Mostly Harmless Econometrics: An Empiricist's Companion. Princeton University Press.

Bertrand, M., Duflo, E., & Mullainathan, S. (2004). How much should we trust differences-in-differences estimates? Quarterly Journal of Economics, 119(1), 249–275.